# Experiment 3b (secondary) · single-band graphs + all-band PSD

Secondary Experiment 3 node features (all bands).

**Runs only this experiment** (`configs/experiments/exp3b_conn_psd_allbands.yaml`): every predefined variant and setting, with nested cross-validation, evaluation and checkpointing. It never starts another experiment.

- **24 variant(s):** `plv__theta__gcn`, `plv__theta__edge_weighted_gat`, `plv__theta__weighted_graphsage`, `plv__alpha1__gcn`, `plv__alpha1__edge_weighted_gat`, `plv__alpha1__weighted_graphsage`, `plv__alpha2__gcn`, `plv__alpha2__edge_weighted_gat`, `plv__alpha2__weighted_graphsage`, `plv__beta__gcn`, `plv__beta__edge_weighted_gat`, `plv__beta__weighted_graphsage`, `ciplv__theta__gcn`, `ciplv__theta__edge_weighted_gat`, `ciplv__theta__weighted_graphsage`, `ciplv__alpha1__gcn`, `ciplv__alpha1__edge_weighted_gat`, `ciplv__alpha1__weighted_graphsage`, `ciplv__alpha2__gcn`, `ciplv__alpha2__edge_weighted_gat`, `ciplv__alpha2__weighted_graphsage`, `ciplv__beta__gcn`, `ciplv__beta__edge_weighted_gat`, `ciplv__beta__weighted_graphsage`
- **Settings:** within_D1, within_D2, pooled, cross_D1_to_D2, cross_D2_to_D1 (cross-dataset only if the DESIGN.md §7 gate passes)
- **Protocol (from the config, unchanged):** 20 repeats × 5-fold subject-level outer CV; inner 4-fold subject-level CV over 4 hyperparameter combination(s); refit on the full training fold; 60 epochs.

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.


Results are saved under `<outputs>/experiments/exp3b_conn_psd_allbands/` on Drive: per-fold/repeat checkpoints,
window and subject predictions, metrics per repeat, selected hyperparameters, summaries, manifests,
and a report folder (results table + **single-experiment diagnostic figures**). The comparative paper
figures across experiments are made separately (`03_comparative_figures.ipynb`).

Requires the graphs from `01_build_graphs.ipynb`.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only if you were asked to switch to newer code
                      # (all notebooks then use the newest commit; unfinished work is recomputed).
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)

## 1. This experiment: progress so far and measured cost on this runtime

In [ ]:
CONFIG = 'configs/experiments/exp3b_conn_psd_allbands.yaml'   # this notebook's experiment
EXPERIMENT = 'exp3b_conn_psd_allbands'
import torch
print('runtime:', ('GPU ' + torch.cuda.get_device_name(0)) if torch.cuda.is_available() else 'CPU only')
if not torch.cuda.is_available():
    print('NOTE: GNN experiment on CPU. It works, but a GPU runtime (Runtime > Change runtime type > T4 GPU) '
          'is much faster. Interrupt now if you want to switch; nothing is lost.')
sh(f"python scripts/experiment.py status --config {CONFIG}")
sh(f"python scripts/experiment.py estimate --config {CONFIG}")

## 2. Run (or resume) this experiment
Saves every completed fold to Drive as it goes. Units that fail are recorded and the others continue.

In [ ]:
try:
    sh(f"python scripts/experiment.py run --config {CONFIG}")
except RuntimeError:
    print('\nWARNING: some units failed or are incompatible; see the table in section 3 and '
          f'{OUT}/experiments/{EXPERIMENT}/_run_status.csv')

## 3. Results of this experiment (saved outputs only)
Subject-level metrics: mean ± SD over CV repeats, subject-bootstrap 95 % CI. Diagnostic figures of this experiment only; not the comparative paper figures. Nothing is selected or ranked here.

In [ ]:
sh(f"python scripts/experiment.py report --config {CONFIG}")
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
report = sorted((Path(OUT) / 'experiments' / EXPERIMENT / '_reports').glob('report_*'))[-1]
print('report folder on Drive:', report)
table = pd.read_csv(report / 'results_summary.csv')
pd.set_option('display.max_rows', 500, 'display.width', 250)
if len(table):
    display(table)
else:
    print('no completed results yet')
for png in sorted((report / 'figures').glob('D[12]_*.png')):
    print(png.name)
    display(Image(filename=str(png)))
print('all files:', sorted(p.name for p in (report / 'figures').glob('*.png')))